# DSE Benchmark v2 — Notebook 0: Mô hình đề xuất & Chuỗi thực nghiệm Ablation (C1–C5)

Notebook này huấn luyện và đánh giá:
1. **So sánh chính (10-fold full trên cả Split B Cold-start và Split A Warm-start)**:
   - `MSSF_v2`: Baseline MSSF đã khử toàn bộ rò rỉ dữ liệu và chọn checkpoint trên `val_select`.
   - `MSSF_plus_full`: Mô hình đề xuất kết hợp đầy đủ 5 đóng góp **C1 + C2 + C3 + C4 + C5**.
2. **Leave-One-Out (LOO) Ablations** (`loo_no_C1`, `loo_no_C2`, `loo_no_C3`, `loo_no_C5`).
3. **Component-wise Ablations** (`c1`, `c2`, `c3`, `c5`) và **Post-hoc Ordinal Conformal Prediction (C4)** trên `val_calib -> test`.

> **Tính năng an toàn cho Colab**: Mọi lệnh đều bật `--resume --drive_dir "$DRIVE_DIR"`. Sau **mỗi fold**, kết quả được sao lưu ngay lập tức lên Google Drive; nếu phiên Colab bị ngắt, chỉ cần chạy lại cell để tiếp tục từ fold kế tiếp.

In [ ]:
# ===== 1. CLONE / UPDATE REPO & MOUNT GOOGLE DRIVE =====
import os, subprocess, torch

REPO_URL = 'https://github.com/hungbuile04/DSE.git'
REPO_BRANCH = 'main'
WORK_DIR = '/content/DSE'

if not os.path.exists(WORK_DIR):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=False)

os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())

# Mount Google Drive for per-fold auto-backup & seamless resume
DRIVE_DIR = '/content/drive/MyDrive/DSE_results_v2'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive mounted. Auto-sync directory: {DRIVE_DIR}')
except Exception as e:
    DRIVE_DIR = '/content/DSE_drive_fallback'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive not mounted ({e}); using local fallback: {DRIVE_DIR}')

os.environ['DRIVE_DIR'] = DRIVE_DIR

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('medium')
    except Exception:
        pass
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: CUDA GPU not detected! Enable Runtime -> Change runtime type -> GPU.')

!pip install -q numpy scipy scikit-learn

In [ ]:
# ===== 2. VERIFY BENCHMARK DATA INTEGRITY (SHA256) =====
!python MSSF_plus/audit/verify_checksums.py --check

---
## 3. Thực nghiệm chính: `MSSF_v2` vs. Mô hình đề xuất (`MSSF_plus_full`) — 10 Folds
Chạy đầy đủ 10 fold (`--mode full`) cho **Split B (Drug Cold-start — trọng tâm bài báo)** và **Split A (Warm-start)**.

In [ ]:
%%time
# Split B (Drug Cold-start) — 10 folds: MSSF_v2 & MSSF_plus_full
!python -m MSSF_plus.scripts.run_ablation \
    --group main \
    --splits B \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

In [ ]:
%%time
# Split A (Warm-start) — 10 folds: MSSF_v2 & MSSF_plus_full
!python -m MSSF_plus.scripts.run_ablation \
    --group main \
    --splits A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 4. Leave-One-Out (LOO) Ablations (`loo_no_C1`, `loo_no_C2`, `loo_no_C3`, `loo_no_C5`)
Đánh giá đóng góp biên của từng thành phần khi gỡ bỏ khỏi mô hình đầy đủ (`10 folds` trên Split B và Split A).

In [ ]:
%%time
!python -m MSSF_plus.scripts.run_ablation \
    --group loo \
    --splits B A \
    --mode full \
    --results_dir ./results_v2 \
    --drive_dir "$DRIVE_DIR" \
    --resume

---
## 5. Component-wise Ablations chi tiết (`c1`, `c2`, `c3`, `c5`)
Mặc định dùng `ABLATION_MODE = 'dev'` (3 folds `0, 1, 2`) để tiết kiệm ngân sách GPU khi sàng lọc biến thể thành phần, hoặc đổi thành `'full'` (10 folds) cho bảng chi tiết cuối cùng.

In [ ]:
%%time
ABLATION_MODE = 'dev'  # Đổi thành 'full' nếu muốn chạy đủ 10 fold cho toàn bộ 20 biến thể con

for grp in ['c1', 'c2', 'c3', 'c5']:
    print(f'\n==================== Running group: {grp} ({ABLATION_MODE}) ====================')
    !python -m MSSF_plus.scripts.run_ablation \
        --group {grp} \
        --splits B \
        --mode {ABLATION_MODE} \
        --results_dir ./results_v2 \
        --drive_dir "$DRIVE_DIR" \
        --resume

---
## 6. Tổng hợp Bảng kết quả, Per-class F1, Conformal C4 & Kiểm định Wilcoxon

In [ ]:
!python -m MSSF_plus.scripts.summarize_results \
    --results_dir "$DRIVE_DIR" \
    --ref_method MSSF_v2 \
    --out_md "$DRIVE_DIR/SUMMARY_PROPOSED_AND_ABLATIONS.md"

!python shared_data/unified_eval.py --results_dir "$DRIVE_DIR" --csv "$DRIVE_DIR/unified_eval_ablation.csv" 